In [1]:
import json, sys, os
import numpy as np
import pandas as pd
from enum import Enum
from scipy.optimize import root_scalar
from typing import Dict, Tuple, Union, Optional

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *

print("Fixed Income Library is loaded.")

Added to sys.path: /Users/nirvana/Documents/QuantBricker
Fixed Income Library is loaded.


### Standard SABR Moments

- Denote SABR's 2-4 moments by : $v$ (variance), $s$ (skew) and $\kappa$ (kurtosis)
- SABR parameters can be expressed in terms of moments:
\begin{align*}\small
&\nu^2T = (\kappa - 3 - \frac{4}{3}s^2)\frac{1}{4}\\[0.75em]
&\rho = \frac{s}{3\sqrt{\nu^2 T}}\\[0.75em]
&\alpha^2T = \frac{v}{1 + \frac{1}{2}\nu^2T}
\end{align*}


In [7]:


def effective_backward_looking_sabr_parameters(
    alpha : float, 
    beta : float, 
    nu : float, 
    rho : float,
    acc_start : float,
    acc_end : float,
    q : Optional[float]=1, # vol decay speed
    apply_transform : Optional[bool]=True
):
    bwd_alpha = alpha
    bwd_nu = nu
    bwd_rho = rho
    bwd_beta = beta
    dalphabwd_dalphafwd = 1.
    dbwdnu_dfwdnu = 1.
    dbwdrho_dfwdrho = 1.
    dbwdalpha_dfwdnu = 0.
    dbwdalpha_dfwdrho = 0.
    dbwdnu_dfwdrho = 0.

    if apply_transform:
        # backward nu
        tau = 2 * q * acc_start + acc_end
        gamma_pre = tau * (2 * tau ** 3 + acc_end ** 3 + q * (4 * q - 2) * (acc_start ** 3) + \
                        6 * q * (acc_start ** 2) * acc_end) / ((4 * q + 3) * (2 * q + 1))
        dgamma_dfwdrho = 6 * q * rho * ((acc_end - acc_start)**2) * \
                    (3 * tau**2 - acc_end**2 + 5 * q * (acc_start**2) + 4 * acc_start * acc_end) / \
                    ((4 * q + 3) * ((3 * q + 2) ** 2))
        gamma = gamma_pre + 0.5 * rho * dgamma_dfwdrho
        dbwdnu_dfwdnu = np.sqrt(gamma * (2 * q + 1) / (tau**3 * acc_end))
        bwd_nu = nu * dbwdnu_dfwdnu
        dbwdnu_dfwdrho = dgamma_dfwdrho * nu * 0.5 * np.sqrt((2 * q + 1) / (tau**3 * acc_end * gamma))

        # backward rho
        dbwdrho_dfwdrho_pre = (3 * tau**2 + 2 * q * acc_start**2 + acc_end**2) / (np.sqrt(gamma) * 6 * q + 4)
        bwd_rho = rho * dbwdrho_dfwdrho_pre
        dbwdrho_dfwdrho = dbwdrho_dfwdrho_pre - 0.5 * bwd_rho / gamma * dgamma_dfwdrho

        # backward alpha
        dh_dfwdnusquare = (tau**2 + 2 * q * (acc_start**2) + acc_end**2) / (2 * acc_end * tau * (q + 1))
        h = nu**2 * dh_dfwdnusquare - bwd_nu**2
        dalphabwd_dalphafwd = np.sqrt(tau / ((2 * q + 1) * acc_end) * np.exp(h * acc_end * 0.5))
        bwd_alpha = alpha * dalphabwd_dalphafwd

        dh_dfwdrho = -2 * bwd_nu * dbwdnu_dfwdrho
        dh_dfwdnu = 2 * nu * dh_dfwdnusquare - 2 * bwd_nu * dbwdnu_dfwdnu
        dbwdalpha_dfwdrho = bwd_alpha * acc_end * 0.25 * dh_dfwdrho
        dbwdalpha_dfwdnu = bwd_alpha * acc_end * 0.25 * dh_dfwdnu

    return (bwd_alpha, bwd_beta, bwd_nu, bwd_rho), \
        (dalphabwd_dalphafwd, dbwdalpha_dfwdnu, dbwdalpha_dfwdrho, dbwdnu_dfwdnu, dbwdnu_dfwdrho, dbwdrho_dfwdrho)


In [8]:
### original parameters
alpha = 0.1108019639023036
beta = 0.6
nu = 0.5
rho = 0.5
shift = 0.04
forward = 0.04
strike = 0.042
tte = 0.5
apply_transform = True
q = 1.

### get backward parameters
transformed_parameters, sensitivities = effective_backward_looking_sabr_parameters(
    alpha, beta, nu, rho, tte, tte + 0.25, q, apply_transform)

transformed_parameters


(0.09774111924989351, 0.6, 0.4417860648704332, 0.5055988628643365)

In [ ]:
from fixedincomelib.date import Date
from fixedincomelib.sabr.sabr_model import SABRModel


class SABRPriceAndRiskCalculator:

    def __init__(self,
                 model : SABRModel,
                 valuation_parameters_collection : ValuationParametersCollection,
                 index : ql.Index,
                 forward : float,
                 strike : float,
                 time_to_expiry : float,
                 tenor : float,
                 call_or_put : CallOrPut,
                 apply_transform : Optional[bool]=False,
                 calc_risk : Optional[bool]=False):
        
        # initialization
        self.model_ = model
        self.vpc_ = valuation_parameters_collection # potential usage
        self.index_ = index
        self.forward_ = forward
        self.strike_ = strike
        self.expiry_ = time_to_expiry
        self.tenor_ = tenor
        self.call_or_put_ = call_or_put
        self.is_overnight_index_ = isinstance(index, ql.OvernightIndex)
        self.calc_risk_ = calc_risk
        self.apply_transform_ = apply_transform

        # resolve negative treatment
        this_component : SABRModelComponent = self.model_.retrieve_model_component(self.index_)
        this_bm : SABRBuildMethod = this_component.build_method
        self.shift_ = this_bm.shift
        self.q_ = 1. # TODO: add vol decay parameter to build method
        self.forward_shifted_ = self.forward_ + self.shift_
        self.strike_shifted_ = self.strike_ + self.shift_

        # time to expiry is quite tricky
        self.effective_expiry_ = self.expiry_
        if self.effective_expiry_ < 0:
            self.effective_expiry_ = self.tenor_

        # get sabr parameters
        # you have to implement get_sabr_parameters for both SABR and its component
        # TODO: self.nv_, self.beta_, self.nu_, self.rho_ = self.model_.get_sabr_parameters(self.index, self.expiry_, self.tenor_)
        self.nv_, self.beta_, self.nu_, self.rho_ = 0.0100, 0.6, 0.2, 0.5
        self.alpha_transform_info_ : Dict[SabrMetrics, float] = SABRAnalytics.alpha_from_atm_normal_sigma(
            self.forward_, 
            self.effective_expiry_,
            self.nv_,
            self.beta_, 
            self.nu_, 
            self.rho_,
            self.shift_,
            self.calc_risk_
        )
        self.alpha_ = self.alpha_transform_info_[SabrMetrics.ALPHA]

        # get effective parameters
        bwd_parameters, self.fwd_to_bwd_sensitivity = effective_backward_looking_sabr_parameters(
            self.alpha_, self.beta_, self.nu_, self.rho_, 
            self.effective_expiry_, 
            self.effective_expiry_ + self.tenor_, 
            self.q, 
            self.apply_transform_)
        self.nv_, self.beta_, self.nu_, self._rho_ = bwd_parameters
    

def calculate_value(self):

    pass

def calculate_risk(self):

    pass

            

        
        


        





In [17]:
this_index = IndexRegistry().get('USD-LIBOR-BBA-3M')
isinstance(this_index, ql.OvernightIndex)

False

In [19]:
this_index = IndexRegistry().get('EUR-USD')
isinstance(this_index, ql.OvernightIndex)

False

In [2]:
# utilities
def display_res(input_dict: Dict[SimpleMetrics, float]):
    display({k.to_string(): v for k, v in input_dict.items()})

### Test all sorts sabr conversions

In [3]:
### hagan's classic formula
alpha = 0.1108019639023036
beta = 0.6
nu = 0.5
rho = 0.5
shift = 0.04
forward = 0.04
strike = 0.042
tte = 0.5

res_imp_log_normal_vol = qfEuropeanOptionSABRLogNormalSigma(
    forward, strike, tte, alpha, beta, rho, nu, shift, True
)
display_res(res_imp_log_normal_vol)

{'implied_log_normal_vol': 0.30976620155710866,
 'd_ln_sigma_d_alpha': 2.783916681139884,
 'd_ln_sigma_d_beta': -0.7728527350756145,
 'd_ln_sigma_d_rho': 0.004816251832454394,
 'd_ln_sigma_d_nu': 0.01789501302461676,
 'd_ln_sigma_d_forward': -2.4033801675345243,
 'd_ln_sigma_d_strike': 0.840061453152265,
 'd_ln_sigma_d_tte': 0.007644740565881988,
 'd_ln_sigma_d_strike_strike': 14.90597064154837}

In [4]:
derivative_keys = [
    SabrMetrics.D_LN_SIGMA_D_FORWARD,
    SabrMetrics.D_LN_SIGMA_D_STRIKE,
    SabrMetrics.D_LN_SIGMA_D_TTE,
    SabrMetrics.D_LN_SIGMA_D_ALPHA,
    SabrMetrics.D_LN_SIGMA_D_BETA,
    SabrMetrics.D_LN_SIGMA_D_RHO,
    SabrMetrics.D_LN_SIGMA_D_NU,
]



In [5]:
### b&r test
key = SimpleMetrics.IMPLIED_LOG_NORMAL_VOL
base_v = res_imp_log_normal_vol[key]

epsilon = 1e-6
bump_vec = [0.0] * 7
for i in range(len(bump_vec)):
    bump_vec[i] += epsilon
    bumped_v = qfEuropeanOptionSABRLogNormalSigma(
        forward + bump_vec[0],
        strike + bump_vec[1],
        tte + bump_vec[2],
        alpha + bump_vec[3],
        beta + bump_vec[4],
        rho + bump_vec[5],
        nu + bump_vec[6],
        shift,
        False,
    )
    # display((bumped_v[key] - base_v) / epsilon)

    analytical_derivative = res_imp_log_normal_vol[derivative_keys[i]]
    numerical_derivative = (bumped_v[key] - base_v) / epsilon
    print(
        f"Param {i} - Numerical: {numerical_derivative:.10f}, Analytical: {analytical_derivative:.10f}, Diff: {numerical_derivative - analytical_derivative:.2e}"
    )
    bump_vec[i] -= epsilon

Param 0 - Numerical: -2.4033521326, Analytical: -2.4033801675, Diff: 2.80e-05
Param 1 - Numerical: 0.8400689060, Analytical: 0.8400614532, Diff: 7.45e-06
Param 2 - Numerical: 0.0076447406, Analytical: 0.0076447406, Diff: 5.24e-11
Param 3 - Numerical: 2.7839168509, Analytical: 2.7839166811, Diff: 1.70e-07
Param 4 - Numerical: -0.7728517557, Analytical: -0.7728527351, Diff: 9.79e-07
Param 5 - Numerical: 0.0048162468, Analytical: 0.0048162518, Diff: -5.04e-09
Param 6 - Numerical: 0.0178950198, Analytical: 0.0178950130, Diff: 6.77e-09


In [6]:
### alpha to atm log-normal sigma
ln_sigma_atm = 0.1108019639023036
beta = 0.6
nu = 0.5
rho = 0.5
shift = 0.04
forward = 0.04
strike = 0.04
tte = 0.5

res_alpha_imp = qfEuropeanOptionSABRAlphaFromATMLogNormalSigma(
    forward, tte, ln_sigma_atm, beta, rho, nu, shift, True
)
display_res(res_alpha_imp)

{'alpha': 0.04,
 'd_alpha_d_ln_sigma_atm': 0.36024000755240065,
 'd_alpha_d_beta': 0.10090124628219954,
 'd_alpha_d_nu': -0.0011936185109739379,
 'd_alpha_d_rho': 0.0004553208990984172,
 'd_alpha_d_forward': 0.19999999999999993,
 'd_alpha_d_tte': -0.0006815089425002967}

In [7]:
### b&r test
key = SabrMetrics.ALPHA
base_v = res_alpha_imp[key]
derivative_alpha_keys = [
    SabrMetrics.D_ALPHA_D_FORWARD,
    SabrMetrics.D_ALPHA_D_TTE,
    SabrMetrics.D_ALPHA_D_LN_SIGMA_ATM,
    SabrMetrics.D_ALPHA_D_BETA,
    SabrMetrics.D_ALPHA_D_RHO,
    SabrMetrics.D_ALPHA_D_NU,
]

epsilon = 1e-6
bump_vec = [0.0] * 6
for i in range(len(bump_vec)):
    bump_vec[i] += epsilon
    bumped_v = qfEuropeanOptionSABRAlphaFromATMLogNormalSigma(
        forward + bump_vec[0],
        tte + bump_vec[1],
        ln_sigma_atm + bump_vec[2],
        beta + bump_vec[3],
        rho + bump_vec[4],
        nu + bump_vec[5],
        shift,
        False,
    )

    analytical_derivative = res_alpha_imp[derivative_alpha_keys[i]]
    numerical_derivative = (bumped_v[key] - base_v) / epsilon
    print(
        f"Param {i} - Numerical: {numerical_derivative:.10f}, Analytical: {analytical_derivative:.10f}, Diff: {numerical_derivative - analytical_derivative:.2e}"
    )
    bump_vec[i] -= epsilon

Param 0 - Numerical: 0.1999992500, Analytical: 0.2000000000, Diff: -7.50e-07
Param 1 - Numerical: -0.0006815089, Analytical: -0.0006815089, Diff: 1.68e-11
Param 2 - Numerical: 0.3602400006, Analytical: 0.3602400076, Diff: -6.99e-09
Param 3 - Numerical: 0.1009013735, Analytical: 0.1009012463, Diff: 1.27e-07
Param 4 - Numerical: 0.0004553215, Analytical: 0.0004553209, Diff: 6.24e-10
Param 5 - Numerical: -0.0011936195, Analytical: -0.0011936185, Diff: -9.91e-10


In [8]:
### alpha from atm normal sigma
sigma_nv = 0.008
beta = 0.6
nu = 0.5
rho = 0.5
forward = 0.04
tte = 0.5
shift = 0.04

res_nv_to_alpha = qfEuropeanOptionSABRAlphaFromATMNormalSigma(
    forward, tte, sigma_nv, beta, rho, nu, shift, True
)
display_res(res_nv_to_alpha)

{'alpha': 0.036115396338483506,
 'd_alpha_d_normal_sigma_atm': 4.507700224131979,
 'd_alpha_d_beta': 0.09111275318884622,
 'd_alpha_d_rho': 0.00042557541277180895,
 'd_alpha_d_nu': -0.0010638495041000173,
 'd_alpha_d_tte': -0.0005857192974776827,
 'd_alpha_d_forward': -0.2701930407085481}

In [9]:
### b&r test
key = SabrMetrics.ALPHA
base_v = res_nv_to_alpha[key]
derivative_keys=[
        SabrMetrics.D_ALPHA_D_FORWARD,
        SabrMetrics.D_ALPHA_D_TTE,
        SabrMetrics.D_ALPHA_D_NORMAL_SIGMA_ATM,
        SabrMetrics.D_ALPHA_D_BETA,
        SabrMetrics.D_ALPHA_D_RHO,
        SabrMetrics.D_ALPHA_D_NU,
    ]

epsilon = 1e-6
bump_vec = [0.0] * 6
for i in range(len(bump_vec)):
    bump_vec[i] += epsilon
    bumped_v = qfEuropeanOptionSABRAlphaFromATMNormalSigma(
        forward + bump_vec[0],
        tte + bump_vec[1],
        sigma_nv + bump_vec[2],
        beta + bump_vec[3],
        rho + bump_vec[4],
        nu + bump_vec[5],
        shift,
        False,
    )
    # display((bumped_v[key] - base_v) / epsilon)
    
    analytical_derivative = res_nv_to_alpha[derivative_keys[i]]
    numerical_derivative = (bumped_v[key] - base_v) / epsilon
    print(
        f"Param {i} - Numerical: {numerical_derivative:.10f}, Analytical: {analytical_derivative:.10f}, Diff: {numerical_derivative - analytical_derivative:.2e}"
    )
    bump_vec[i] -= epsilon

Param 0 - Numerical: -0.2701903443, Analytical: -0.2701930407, Diff: 2.70e-06
Param 1 - Numerical: -0.0005857192, Analytical: -0.0005857193, Diff: 8.82e-11
Param 2 - Numerical: 4.5076994853, Analytical: 4.5077002241, Diff: -7.39e-07
Param 3 - Numerical: 0.0911128681, Analytical: 0.0911127532, Diff: 1.15e-07
Param 4 - Numerical: 0.0004255760, Analytical: 0.0004255754, Diff: 5.58e-10
Param 5 - Numerical: -0.0010638504, Analytical: -0.0010638495, Diff: -8.93e-10


In [10]:
### atm normal sigma from alpha
# from alpha to normal
alpha = res_nv_to_alpha[SabrMetrics.ALPHA]
beta = 0.6
nu = 0.5
rho = 0.5
forward = 0.04
tte = 0.5
shift = 0.04
res_alpha_to_nv = qfEuropeanOptionSABRAlphaFromATMNormalSigma(
    forward, tte, alpha, beta, rho, nu, shift, True
)
display(res_alpha_to_nv)

{<SabrMetrics.ALPHA: 'alpha'>: 0.16255319957848316,
 <SabrMetrics.D_ALPHA_D_NORMAL_SIGMA_ATM: 'd_alpha_d_normal_sigma_atm'>: 4.496739708984385,
 <SabrMetrics.D_ALPHA_D_BETA: 'd_alpha_d_beta'>: 0.4088799778660368,
 <SabrMetrics.D_ALPHA_D_RHO: 'd_alpha_d_rho'>: -0.0001769999307616879,
 <SabrMetrics.D_ALPHA_D_NU: 'd_alpha_d_nu'>: -0.0067832755879983,
 <SabrMetrics.D_ALPHA_D_TTE: 'd_alpha_d_tte'>: -0.0035433005515043874,
 <SabrMetrics.D_ALPHA_D_FORWARD: 'd_alpha_d_forward'>: -1.2172527431973226}

In [11]:
### b&r test
key = SabrMetrics.ALPHA
base_v = res_alpha_to_nv[key]
derivative_keys=[
        SabrMetrics.D_ALPHA_D_FORWARD,
        SabrMetrics.D_ALPHA_D_TTE,
        SabrMetrics.D_ALPHA_D_NORMAL_SIGMA_ATM,
        SabrMetrics.D_ALPHA_D_BETA,
        SabrMetrics.D_ALPHA_D_RHO,
        SabrMetrics.D_ALPHA_D_NU,
    ]

epsilon = 1e-6
bump_vec = [0.0] * 6
for i in range(len(bump_vec)):
    bump_vec[i] += epsilon
    bumped_v = qfEuropeanOptionSABRAlphaFromATMNormalSigma(
        forward + bump_vec[0],
        tte + bump_vec[1],
        alpha + bump_vec[2],
        beta + bump_vec[3],
        rho + bump_vec[4],
        nu + bump_vec[5],
        shift,
        False,
    )
    analytical_derivative = res_alpha_to_nv[derivative_keys[i]]
    numerical_derivative = (bumped_v[key] - base_v) / epsilon
    print(
        f"Param {i} - Numerical: {numerical_derivative:.10f}, Analytical: {analytical_derivative:.10f}, Diff: {numerical_derivative - analytical_derivative:.2e}"
    )
    # display((bumped_v[key] - base_v) / epsilon)
    bump_vec[i] -= epsilon

Param 0 - Numerical: -1.2172376726, Analytical: -1.2172527432, Diff: 1.51e-05
Param 1 - Numerical: -0.0035435636, Analytical: -0.0035433006, Diff: -2.63e-07
Param 2 - Numerical: 4.4967327758, Analytical: 4.4967397090, Diff: -6.93e-06
Param 3 - Numerical: 0.4088804915, Analytical: 0.4088799779, Diff: 5.14e-07
Param 4 - Numerical: -0.0001769975, Analytical: -0.0001769999, Diff: 2.46e-09
Param 5 - Numerical: -0.0067832794, Analytical: -0.0067832756, Diff: -3.77e-09
